<a href="https://colab.research.google.com/github/saitejamudapalli/Project-HealthCare-Provider-Analysis/blob/main/BRONZE_LAYER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#CODE FOR BRONZE_LAYER


from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
from datetime import datetime
import os

# ===== CONFIG =====
PROJECT_ID = "even-blueprint-441418-p2"
DATASET_ID = "BRONZE_LAYER"
TABLE_ID = "PATIENTS_BRONZE"
CSV_PATH = r"/content/patients.csv"
KEY_PATH = r"/content/even-blueprint-441418-p2-043f8a9d855b.json(KEY).json"  # service account key


# ===== AUTHENTICATION =====
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

# ===== CREATE DATASET IF NOT EXISTS =====
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
try:
    client.get_dataset(dataset_ref)
    print(f"✅ Dataset already exists: {dataset_ref}")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"  # Change to your region if needed
    client.create_dataset(dataset, exists_ok=True)
    print(f"✅ Created dataset: {dataset_ref}")

# ===== LOAD RAW DATA =====
print("📥 Reading CSV file...")
df_raw = pd.read_csv(CSV_PATH)

# Add metadata columns
df_raw["_ingest_time_utc"] = datetime.utcnow().isoformat()
df_raw["_source_file"] = os.path.basename(CSV_PATH)
df_raw["_raw_row_id"] = range(1, len(df_raw) + 1)

print(f"✅ Loaded {len(df_raw)} rows from CSV.")

# ===== UPLOAD TO BIGQUERY =====
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # Replace table data
)

print(f"🚀 Loading data into BigQuery table: {table_ref} ...")
job = client.load_table_from_dataframe(df_raw, table_ref, job_config=job_config)
job.result()  # Wait for the job to complete

print(f"✅ Bronze layer loaded successfully to {table_ref}")